In [ ]:
# SINTAXIS CAMBIA DEL SELECT
import oracledb
try:
    conn = oracledb.connect(
        user="hr",
        password="basededatos",
        dsn="localhost:1521/XE"  
    )
    cursor = conn.cursor()
    cursor.callproc("dbms_output.enable")

except oracledb.Error as e:
    print(f"Error detectado: {e}")

In [7]:
try:
    cursor.execute("""
    DECLARE 
        v_start         NUMBER(7) := 110;
    BEGIN
        IF      v_start > 100 THEN
                v_start := .2 * v_start;
        ELSIF  v_start >= 50 THEN
                v_start := .5 * v_start;
        ELSE    
                v_start := .1 * v_start;
        END IF;
        DBMS_OUTPUT.PUT_LINE('Valor de Start: ' || TO_CHAR(v_start));
    END;
    """)
    line = cursor.var(str)
    status = cursor.var(int)
    
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(f"Error detectado: {e}")

Valor de Start: 22


In [13]:
# SINTAXIS CAMBIA DEL SELECT
import oracledb
try:
    cursor.execute("""
    DECLARE
        v_salary            employees.employee_id%TYPE;
        v_salario           VARCHAR2(10);
    BEGIN
        SELECT       salary
        INTO         v_salary
        FROM         employees
        WHERE        employee_id = 101;
        IF      v_salary BETWEEN 0 AND 1500 THEN
                v_salario := 'bajo';
        ELSIF  v_salary BETWEEN 1501 AND 2500 THEN
                v_salario := 'medio';
        ELSE    
                v_salario := 'alto';
        END IF;
        DBMS_OUTPUT.PUT_LINE('El salario es: ' || v_salario);
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(f"Error detectado: {e}")



El salario es: alto


In [16]:
# SINTAXIS CAMBIA DEL SELECT
import oracledb
try:
    cursor.execute("""
    DECLARE
        v_salary            employees.employee_id%TYPE;
        v_salario           VARCHAR2(10);
    BEGIN
        SELECT       
            CASE 
                WHEN salary BETWEEN 0 AND 1500 THEN 'bajo'
                WHEN salary BETWEEN 1501 AND 2500 THEN 'medio'
                ELSE 'alto'
            END
        INTO         v_salario
        FROM         employees
        WHERE        employee_id = 101;
        DBMS_OUTPUT.PUT_LINE('El salario es: ' || v_salario);
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(f"Error detectado: {e}")



El salario es: alto


In [ ]:
# SINTAXIS CAMBIA DEL SELECT
import oracledb
try:
    employee_number = 101
    # cursor.execute("""
    # CREATE TABLE retired_emp AS SELECT * FROM employees
    # """)
    # cursor.execute("""
    # COMMIT
    # """)
    cursor.execute(f"""
    DECLARE
        emp_rec             employees%ROWTYPE;
    BEGIN
        SELECT * INTO emp_rec FROM employees
        WHERE employee_id = {employee_number};
        INSERT INTO retired_emp(employee_id, first_name, last_name, job_id, manager_id,
                                 hire_date, email, salary, department_id)
        VALUES (emp_rec.employee_id,emp_rec.first_name, emp_rec.last_name, emp_rec.job_id, emp_rec.manager_id,
                emp_rec.hire_date, emp_rec.email, emp_rec.salary, emp_rec.department_id);
        COMMIT;
    END;
    """)
    cursor.execute(f"""
    SELECT * FROM retired_emp WHERE employee_id = {employee_number}
    """)
    results = cursor.fetchall()
    for row in results:
        print(row)
    
    line = cursor.var(str)
    status = cursor.var(int)
    
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(f"Error detectado: {e}")



(101, 'Neena', 'Kochhar', 'neena.kochhar@sqltutorial.org', '515.123.4568', datetime.datetime(1989, 9, 21, 0, 0), 5, 17000.0, 100, 9)
(101, 'Neena', 'Kochhar', 'neena.kochhar@sqltutorial.org', None, datetime.datetime(1989, 9, 21, 0, 0), 5, 17000.0, 100, 9)


Cursores: concepto de cursor.
Es útil, no puedo abusar del mismo, implicaciones de memoria
Todos los cursores que ejecute SE VAN A MEMORIA
¿Qué es?
Una tabla T, etngo columnas, {t1, t2, ..., tn}
Puedo crear un cursor escogiendo ciertas columnas de una tabla, 
O escoger de varias tablas, con la finalidad de ponder manipular datos 
en las tablas.

Para que no se me desborde la memori debo seguir pasos especificos

Cursor explícitos, propios del programador, 
- Declarar el Cursor PL
- Abrir el cursor OPEN
- Fetch: cargo las actuales filas a las variables, ejecutar varias veces
con un loop
- CLOSE ya hice una operacion, lo 
- Si lo necesitaré no lo cierro, lo vuelvo a utilizar.

Declaracion: 

CURSOR cursor_name IS 
    select_statement;
    


In [ ]:
# DECLARE
#     CURSOR emp_cursor IS
#         SELECT employee_id, last_name
#         FROM employees;
#     CURSOR emp_cursor IS
#         SELECT employee_id, last_name
#         FROM employees;
#                  
# FETCH cursor_name INTO (varaible1, variable2, ...)
#                -- o record name
# CLOSE

# ISOPEN, evalua si el cursor esta abierto
#

In [27]:

try:
    cursor.execute("""
    DECLARE
        v_empno         employees.employee_id%TYPE;
        v_ename         employees.last_name%TYPE;
        CURSOR emp_cursor IS
            SELECT employee_id, last_name
            FROM employees;
    BEGIN
        OPEN emp_cursor;
        LOOP
            FETCH emp_cursor INTO v_empno, v_ename;
            EXIT WHEN emp_cursor%ROWCOUNT > 10 OR emp_cursor%NOTFOUND;
            DBMS_OUTPUT.PUT_LINE(TO_CHAR(v_empno) || '     ' || v_ename);
        END LOOP;
        CLOSE emp_cursor;
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(f"Error detectado: {e}")



100     King
101     Kochhar
102     De Haan
103     Hunold
104     Ernst
105     Austin
106     Pataballa
107     Lorentz
108     Greenberg
109     Faviet


In [ ]:

try:
    cursor.execute("""
    DECLARE
        CURSOR emp_cursor IS
            SELECT employee_id, last_name
            FROM employees;
    BEGIN
        OPEN emp_cursor;
        LOOP
            FETCH emp_cursor INTO v_empno, v_ename;
            EXIT WHEN emp_cursor%ROWCOUNT > 10 OR emp_cursor%NOTFOUND;
            DBMS_OUTPUT.PUT_LINE(TO_CHAR(v_empno) || '     ' || v_ename);
        END LOOP;
        CLOSE emp_cursor;
    END;
    """)

    line = cursor.var(str)
    status = cursor.var(int)
    
    while True:
        cursor.callproc("dbms_output.get_line", (line, status))
        if status.getvalue() != 0:
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(f"Error detectado: {e}")

